# Lab 2 · Data Analyst Pipeline: plain English → SQL → answer

Most of a company's knowledge is not in documents but in **tables**: sales, customers, invoices, tickets. Lab 1 gave the model documents to read; here we give it a **database to query**. You will build a text-to-SQL pipeline whose every step is visible, on a real sales database with 11 tables, then point the same pipeline at your own spreadsheet.

```
question ─▶ 1 catalogue the tables ─▶ 2 choose the relevant ones (model) ─▶ 3 inject their schema
         ─▶ 4 write SQL (model) ─▶ 5 validate + run ─▶ 6 error? feed it back and retry ─▶ 7 explain the result (model)
```

Libraries do the plumbing: **LangChain's `SQLDatabase`** inspects tables and formats schemas, **`sqlglot`** parses the generated SQL, **pandas** runs it and shows results. The model makes exactly three decisions: which tables, which SQL, what the numbers mean.

### Setup: Runtime → Change runtime type → **T4 GPU**, then run this cell 

In [1]:
!pip install -q -U transformers accelerate langchain-community sqlglot openpyxl

import torch, json, re
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"          # 4B open model, ~9 GB in fp16, fits a free T4. (Fallback: "Qwen/Qwen3-4B")
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(MODEL_ID, dtype=torch.float16, device_map="auto")

def chat(messages, max_new_tokens=400, temperature=0.0):
    """The one function that talks to the model: a prompt string or a list of {role, content} -> reply text."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=temperature > 0,
                         temperature=temperature if temperature > 0 else None)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()

print(chat("Say hello in five words."))

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 103.5 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 35.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 83.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 756.2/756.2 kB 54.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 62.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.1/73.1 kB 8.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.
ibis-framework 9.5.0 requires sqlglot<25.21,>=23.4, but you have sqlglot 30.19.0 which is incompatible.


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:138: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
  warnings.warn(f"\nError while fetching `HF_TOKEN` secret value from your vault: '{str(e)}'.")


config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/6.72M [00:00<?, ?B/s]

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


Hello there, friend. How are you?


## 0 · The problem: the model has no hands

In [2]:
print(chat("In our company's sales database, which three countries bring in the most revenue?"))   # it cannot know - or worse, it guesses

I don't have access to your company's sales database, so I can't tell you which three countries bring in the most revenue. To find this information, you would need to:

1. Query your database using SQL (or your preferred tool) with a query like:
   ```sql
   SELECT country, SUM(revenue) as total_revenue
   FROM sales
   GROUP BY country
   ORDER BY total_revenue DESC
   LIMIT 3;
   ```
2. Check your business intelligence dashboard (e.g., Tableau, Power BI, Looker) if one is set up.
3. Export and analyze the sales data directly if you have access to the raw files.

If you can share a sample of your data structure or provide anonymized data, I'd be happy to help craft the exact query or analysis!


## 1 · The data: a real sales database

[Chinook](https://github.com/lerocha/chinook-database) is a digital music store: customers, invoices, invoice lines, tracks, albums, artists, genres, employees. Eleven related tables in one 1 MB SQLite file, exactly the shape of a small company's ERP or CRM export.

`connect()` is how the pipeline is pointed at *any* SQLite file: a connection for pandas, an inspector for LangChain, and the compact catalogue for step 1. The same call is used later for your own spreadsheet.

In [3]:
import urllib.request, sqlite3, pandas as pd
from IPython.display import display
from langchain_community.utilities import SQLDatabase

def step(n, title, *lines):
    '''One banner per pipeline step, so every run reads the same way.'''
    print(f"\n{'━' * 72}\n Step {n} · {title}\n{'━' * 72}")
    for line in lines:
        print(line)

def connect(path):
    '''Point the pipeline at a SQLite file -> (connection for pandas, inspector for LangChain, compact catalogue).'''
    conn = sqlite3.connect(path)
    db = SQLDatabase.from_uri(f"sqlite:///{path}", sample_rows_in_table_info=3)          # full schema + sample rows, per table
    catalogue = SQLDatabase.from_uri(f"sqlite:///{path}", sample_rows_in_table_info=0).table_info   # all tables, no rows: compact
    step(1, "Catalogue", f"{len(db.get_usable_table_names())} tables, {len(catalogue):,} characters: {db.get_usable_table_names()}")
    return conn, db, catalogue

urllib.request.urlretrieve("https://github.com/lerocha/chinook-database/releases/download/v1.4.5/Chinook_Sqlite.sqlite", "chinook.db")
conn, db, catalogue = connect("chinook.db")
pd.read_sql("SELECT * FROM Invoice LIMIT 5", conn)

/tmp/ipykernel_1405/1714776065.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.utilities import SQLDatabase



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 1 · Catalogue
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
11 tables, 3,192 characters: ['Album', 'Artist', 'Customer', 'Employee', 'Genre', 'Invoice', 'InvoiceLine', 'MediaType', 'Playlist', 'PlaylistTrack', 'Track']


,InvoiceId,CustomerId,InvoiceDate,BillingAddress,BillingCity,BillingState,BillingCountry,BillingPostalCode,Total
0,1,2,2009-01-01 00:00:00,Theodor-Heuss-Straße 34,Stuttgart,None,Germany,70174,1.98
1,2,4,2009-01-02 00:00:00,Ullevålsveien 14,Oslo,None,Norway,0171,3.96
2,3,8,2009-01-03 00:00:00,Grétrystraat 63,Brussels,None,Belgium,1000,5.94
3,4,14,2009-01-06 00:00:00,8210 111 ST NW,Edmonton,AB,Canada,T6G 2C7,8.91
4,5,23,2009-01-11 00:00:00,69 Salem Street,Boston,MA,USA,2113,13.86


## Step 1 · Catalogue the tables

The library reads the database's own description of itself: every table as a `CREATE TABLE` statement, with column types, primary keys and **foreign keys**. The foreign keys are what will let the model join tables correctly. The catalogue is built once, in `connect()`.

In [4]:
print(catalogue[:2000], "...")


CREATE TABLE "Album" (
	"AlbumId" INTEGER NOT NULL, 
	"Title" NVARCHAR(160) NOT NULL, 
	"ArtistId" INTEGER NOT NULL, 
	PRIMARY KEY ("AlbumId"), 
	FOREIGN KEY("ArtistId") REFERENCES "Artist" ("ArtistId")
)


CREATE TABLE "Artist" (
	"ArtistId" INTEGER NOT NULL, 
	"Name" NVARCHAR(120), 
	PRIMARY KEY ("ArtistId")
)


CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)


CREATE TABLE "Employee" (
	"EmployeeId" INTEGER NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"FirstName" NVARCHAR(20) NOT NULL, 
	"Title" NVARCHAR(30), 
	"ReportsTo" INTEGER, 
	"BirthDate" 

## Step 2: Choose the tables for the question, the model's first decision

Eleven schemas with sample rows would be thousands of tokens per question, most of them irrelevant. So the model first reads the compact catalogue and names the tables the question needs, and says why. We keep only names that really exist.

In [5]:
def chat_json(messages, **kw):
    """Same as chat(), but returns the first JSON object found in the reply as a dict."""
    text = chat(messages, **kw)
    match = re.search(r"\{.*\}", text, re.S)
    return json.loads(match.group()) if match else {}

In [6]:
def choose_tables(question):
    '''The model reads the catalogue (names, columns, keys) and picks the tables this question needs -> (tables, why).'''
    reply = chat_json(f"Database catalogue:\n{catalogue}\n\nQuestion: {question}\n\n"
                      f'Which tables are needed to answer it? Usually one to three, plus any table needed to join them. '
                      f'Return JSON: {{"tables": [...], "why": "..."}}')
    tables = [t for t in reply.get("tables", []) if t in db.get_usable_table_names()] or list(db.get_usable_table_names())
    return tables, reply.get("why", "")

QUESTION = "Which five countries bring in the most revenue, and how much each?"
tables, why = choose_tables(QUESTION)
step(2, "Choose tables (model)", f"tables: {tables}", f"why: {why}")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 2 · Choose tables (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tables: ['Invoice', 'Customer']
why: To determine the countries with the most revenue, we need the 'Total' amount from the 'Invoice' table and the 'Country' information from the 'Customer' table. We join these two tables on 'CustomerId' to link each invoice total to the customer's country, then group by country and sum the totals.


## Step 3 · Inject the schema of just those tables

Now the library formats the chosen tables in full: the `CREATE TABLE` statement **plus three sample rows**, so the model sees real values (date formats, country spellings, currency scale) and not only column names.

In [7]:
schema = db.get_table_info(tables)
step(3, "Inject schema", f"{len(tables)} tables, {len(schema):,} characters: CREATE TABLE + 3 sample rows each")
print(schema)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 3 · Inject schema
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2 tables, 1,851 characters: CREATE TABLE + 3 sample rows each

CREATE TABLE "Customer" (
	"CustomerId" INTEGER NOT NULL, 
	"FirstName" NVARCHAR(40) NOT NULL, 
	"LastName" NVARCHAR(20) NOT NULL, 
	"Company" NVARCHAR(80), 
	"Address" NVARCHAR(70), 
	"City" NVARCHAR(40), 
	"State" NVARCHAR(40), 
	"Country" NVARCHAR(40), 
	"PostalCode" NVARCHAR(10), 
	"Phone" NVARCHAR(24), 
	"Fax" NVARCHAR(24), 
	"Email" NVARCHAR(60) NOT NULL, 
	"SupportRepId" INTEGER, 
	PRIMARY KEY ("CustomerId"), 
	FOREIGN KEY("SupportRepId") REFERENCES "Employee" ("EmployeeId")
)

/*
3 rows from Customer table:
CustomerId	FirstName	LastName	Company	Address	City	State	Country	PostalCode	Phone	Fax	Email	SupportRepId
1	Luís	Gonçalves	Embraer - Empresa Brasileira de Aeronáutica S.A.	Av. Brigadeiro Faria Lima, 2170	São José dos Campos	SP	Brazil	12227-000	+5

## Step 4 · Write the SQL — the model's second decision

The prompt is the schema, the question and a few rules. The model answers with a ```sql block; we cut it out with one regex. Nothing runs yet. On a retry the prompt also carries the failed query and the database's error.

In [8]:
def write_sql(question, schema, error=None, failed_sql=None, temperature=0.0):
    prompt = (f"You write {db.dialect} SQL.\n\nSchema:\n{schema}\n\nQuestion: {question}\n\n"
              f"Rules: one SELECT statement; use only tables and columns that appear in the schema, spelled exactly; "
              f"join through the foreign keys; if you use an aggregate, every column you select or order by that is not "
              f"itself aggregated must appear in GROUP BY; "
              f"LIMIT 20 unless the question asks for a count. Reply with the SQL in a ```sql block.")
    if error:
        prompt += f"\n\nYour previous query:\n{failed_sql}\n\nfailed with this error:\n{error}\n\nWrite a corrected query."
    reply = chat(prompt, max_new_tokens=300, temperature=temperature)
    block = re.search(r"```sql\s*(.*?)```", reply, re.S | re.I)
    return (block.group(1) if block else reply).strip().rstrip(";")

sql = write_sql(QUESTION, schema)
step(4, "Write SQL (model)", sql)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 4 · Write SQL (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECT 
    c.Country,
    SUM(i.Total) AS TotalRevenue
FROM "Invoice" i
JOIN "Customer" c ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY TotalRevenue DESC
LIMIT 5


## Step 5 · Validate, then run

Two checks before any result: `sqlglot` parses the query and confirms it is a **SELECT** (a generated `DROP TABLE` must never reach the database), then pandas executes it and returns a table. If the database rejects the query, we keep its error message: that is step 6's input.

In [9]:
import sqlglot

def run_sql(sql):
    '''Parse with sqlglot (must be a SELECT), then execute with pandas.'''
    tree = sqlglot.parse_one(sql, read=db.dialect)
    if not isinstance(tree, (sqlglot.exp.Select, sqlglot.exp.Union)):
        raise ValueError("only SELECT statements are allowed")
    return pd.read_sql(sql, conn)

def db_error(e):
    '''The reason, not the echo: pandas prepends the whole query to the database's message, the diagnosis is the last line.'''
    return str(e.__cause__ or e).strip().splitlines()[-1]

try:
    result = run_sql(sql)
    step(5, "Validate + run", "SELECT only ✓", f"{len(result)} rows × {len(result.columns)} columns"); display(result)
except Exception as e:
    step(6, "Retry", f"database says: {db_error(e)}")                                    # on to step 6


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 5 · Validate + run
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECT only ✓
5 rows × 2 columns


,Country,TotalRevenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


## Step 6 · When the SQL fails, feed the error back and try again

Generated SQL fails often: a misspelt column, a wrong join, a date function the dialect does not have. The database says exactly why, and that message goes back into the prompt together with the failed query. First a deliberate mistake, so everyone sees the repair; then the loop that does it automatically. The first attempt is greedy (temperature 0); retries add a little randomness so the model does not repeat itself.

In [10]:
broken = '''SELECT BillingCountry, SUM(Totals) AS revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY revenue DESC LIMIT 5'''
try:
    run_sql(broken)
except Exception as e:
    error = db_error(e)
    step(6, "Retry", f"database says: {error}", "→ the failed query and this message go back into the prompt")
    step(4, "Write SQL (model) · attempt 2", write_sql(QUESTION, schema, error=error, failed_sql=broken))


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 6 · Retry
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
database says: no such column: Totals
→ the failed query and this message go back into the prompt

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 4 · Write SQL (model) · attempt 2
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECT BillingCountry, SUM(Total) AS revenue
FROM Invoice
GROUP BY BillingCountry
ORDER BY revenue DESC
LIMIT 5


In [11]:
def query(question, schema, max_attempts=3):
    '''Steps 4-6: write SQL, validate + run, feed the error back, until it works. Returns (sql, result) or (sql, None).'''
    sql, error = None, None
    for attempt in range(1, max_attempts + 1):
        sql = write_sql(question, schema, error, failed_sql=sql, temperature=0.0 if attempt == 1 else 0.3)   # greedy once, then vary
        step(4, f"Write SQL (model) · attempt {attempt}", sql or "(the model returned no SQL)")
        try:
            result = run_sql(sql)
            step(5, "Validate + run", "SELECT only ✓", f"{len(result)} rows × {len(result.columns)} columns"); display(result.head(20))
            return sql, result
        except Exception as e:
            error = db_error(e)
            step(6, "Retry", f"database says: {error}", "→ the failed query and this message go back into the prompt")
    return sql, None

sql, result = query(QUESTION, schema)


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 4 · Write SQL (model) · attempt 1
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECT 
    c.Country,
    SUM(i.Total) AS TotalRevenue
FROM "Invoice" i
JOIN "Customer" c ON i.CustomerId = c.CustomerId
GROUP BY c.Country
ORDER BY TotalRevenue DESC
LIMIT 5

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 5 · Validate + run
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
SELECT only ✓
5 rows × 2 columns


,Country,TotalRevenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


## Step 7 · Explain the result in plain English — the model's third decision

A table of numbers is not an answer. The model gets the question, the SQL and the result, and writes two sentences for a manager, using only numbers that are in the table.

In [12]:
def explain(question, sql, result):
    return chat(f"Question: {question}\nSQL used: {sql}\nResult table:\n{result.head(20).to_string(index=False)}\n\n"
                f"Answer the question in two or three sentences for a manager. Use only numbers that appear in the table.", max_new_tokens=200)

step(7, "Explain (model)", explain(QUESTION, sql, result) if result is not None else "no result to explain - the query never worked")


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 7 · Explain (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The five countries generating the highest revenue are the USA, Canada, France, Brazil, and Germany, with totals of $523.06, $303.96, $195.10, $190.10, and $156.48 respectively. The USA leads significantly in revenue, followed by Canada, while France and Brazil contribute nearly equal amounts.


## The whole pipeline in one function

Steps 1 to 7 in order: the catalogue from `connect()` (1), `choose_tables` (2), `get_table_info` (3), `query` (4 to 6), `explain` (7). The code owns the order; the model owns the three decisions. Every step prints a banner with what it decided, so a wrong answer can be traced to the step that caused it.

In [13]:
def ask(question):
    print(f"\nQUESTION: {question}")
    step(1, "Catalogue", f"{len(db.get_usable_table_names())} tables available (built once in connect())")
    tables, why = choose_tables(question);      step(2, "Choose tables (model)", f"tables: {tables}", f"why: {why}")
    schema = db.get_table_info(tables);         step(3, "Inject schema", f"{len(tables)} tables, {len(schema):,} characters: CREATE TABLE + 3 sample rows each")
    sql, result = query(question, schema)       # steps 4-6 print themselves
    if result is None:
        step(7, "Explain (model)", "No working query after three attempts - hand to a human."); return
    step(7, "Explain (model)", explain(question, sql, result))

ask("Which three genres sell the most tracks?")


QUESTION: Which three genres sell the most tracks?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 1 · Catalogue
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
11 tables available (built once in connect())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 2 · Choose tables (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tables: ['Genre', 'Track']
why: To determine which three genres sell the most tracks, we need the 'Genre' table to identify unique genre names and the 'Track' table to count the number of tracks associated with each genre. By joining these tables on 'GenreId', we can group the tracks by genre and order them by count to find the top three.

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 3 · Inject schema
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
2 tables, 1,035 characters: CREATE TABLE + 3

,Genre,TrackCount
0,Rock,1297
1,Latin,579
2,Metal,374



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 7 · Explain (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
The three genres that sell the most tracks are Rock with 1,297 tracks, followed by Latin with 579 tracks, and Metal with 374 tracks. These figures represent the top three performers based on the provided sales data.


In [14]:
ask("Which employee looks after the customers who spent the most in total?")     # needs Employee, Customer, Invoice
ask("How did total revenue change from year to year?")                            # needs a date function


QUESTION: Which employee looks after the customers who spent the most in total?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 1 · Catalogue
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
11 tables available (built once in connect())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 2 · Choose tables (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tables: ['Customer', 'Invoice', 'Employee']
why: To find the employees who look after customers who spent the most, we need to join the 'Customer' table (to link customers to their support representative via 'SupportRepId'), the 'Invoice' table (to calculate total spending per customer), and the 'Employee' table (to retrieve the employee names/titles associated with the support representative IDs). The 'Customer' table provides the link between customers and employees, while 'Invoice' provides the spending data.

━━━━━━━━

,LastName,FirstName
0,Peacock,Jane
1,Park,Margaret
2,Johnson,Steve



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 7 · Explain (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Jane Peacock is the employee who looks after the customers who spent the most in total, followed by Margaret Park and Steve Johnson. The query ranks these employees based on the sum of their customers' invoice totals in descending order.

QUESTION: How did total revenue change from year to year?

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 1 · Catalogue
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
11 tables available (built once in connect())

━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 2 · Choose tables (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
tables: ['Invoice', 'InvoiceLine']
why: To calculate total revenue, we need the 'Invoice' table to identify individual invoices and their 

,Year,TotalRevenue
0,2009,449.46
1,2010,481.45
2,2011,469.58
3,2012,477.53
4,2013,450.58



━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 7 · Explain (model)
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
Total revenue increased from $449.46 in 2009 to a peak of $481.45 in 2010, before declining to $469.58 in 2011 and rising slightly to $477.53 in 2012. The revenue then dropped to $450.58 in 2013, which is the lowest figure recorded in this dataset.


## The same pipeline in LangChain

Steps 1 to 4 of this lab exist inside LangChain as one function, `create_sql_query_chain`: it reads the catalogue, injects the schema with sample rows, prompts the model and returns the SQL. To use it with our model, LangChain needs one small adapter, a class with a single method that calls `chat()`. That adapter is the whole trick for plugging any model into the framework.

Read the prompt it hides. Then run its SQL through our own `run_sql` and compare with the query the pipeline wrote.

In [15]:
from langchain_core.language_models.llms import LLM
try:
    from langchain_classic.chains import create_sql_query_chain            # LangChain 1.x keeps the classic chains here
except ImportError:
    from langchain.chains import create_sql_query_chain                    # LangChain 0.3

class LocalModel(LLM):
    """The adapter: LangChain calls _call(prompt), we call chat(). Nothing else is needed."""
    def _call(self, prompt, stop=None, run_manager=None, **kwargs):
        text = chat(prompt, max_new_tokens=300)
        for s in stop or []:
            text = text.split(s)[0]                                         # honour the stop sequences the chain asks for
        return text
    @property
    def _llm_type(self):
        return "qwen-local"

llm = LocalModel()
chain = create_sql_query_chain(llm, db)                                    # catalogue -> schema -> prompt -> model -> SQL, packaged
print(chain.get_prompts()[0].template[:900], "...")                       # the prompt the framework hides: steps 1-4 are in there

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in double quotes (") to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use date('now') function to get the current date, if the question involves ...


In [16]:
raw = chain.invoke({"question": QUESTION})
found = re.search(r"```sql\s*(.*?)```", raw, re.S | re.I) or re.search(r"(SELECT[\s\S]*?)(?=\n\s*\n|SQLResult|$)", raw, re.I)
sql_langchain = (found.group(1) if found else raw).strip().rstrip(";")
print(sql_langchain)
display(run_sql(sql_langchain))                                             # its SQL, our validation and execution

SELECT 
    c."Country",
    SUM(i."Total") AS TotalRevenue
FROM "Invoice" i
JOIN "Customer" c ON i."CustomerId" = c."CustomerId"
GROUP BY c."Country"
ORDER BY TotalRevenue DESC
LIMIT 5


,Country,TotalRevenue
0,USA,523.06
1,Canada,303.96
2,France,195.10
3,Brazil,190.10
4,Germany,156.48


In [17]:
from langchain_community.agent_toolkits import SQLDatabaseToolkit

for tool in SQLDatabaseToolkit(db=db, llm=llm).get_tools():               # the four tools LangChain's SQL agent is given
    print(f"{tool.name:<24} {tool.description[:110]}")

sql_db_query             Input to this tool is a detailed and correct SQL query, output is a result from the database. If the query is 
sql_db_schema            Input to this tool is a comma-separated list of tables, output is the schema and sample rows for those tables.
sql_db_list_tables       Input is an empty string, output is a comma-separated list of tables in the database.
sql_db_query_checker     Use this tool to double check if your query is correct before executing it. Always use this tool before execut


Same steps, different surface. The chain saved about thirty lines and brings a prompt tuned over many databases; it also hid the catalogue step and the choice of tables, which is why the pipeline you built is the version to debug and the chain is the version to ship once you trust it. The toolkit's four tools are our steps 1, 3, 5 and 6 with names: list tables, show a schema, run a query and return errors as text, check a query with the model. Hand them to the agent loop from the Interactive lab and you have LangChain's SQL agent.

**Pipeline, not agent.** The steps and their order are fixed in `ask()`; the model only fills in three of them. That is the right shape when you know the path in advance: cheaper, predictable, and every failure has an address. Lab 3 shows the other shape, where the model also decides *whether* to take another step.

## 🗂️ Your turn: your own spreadsheet

Upload an Excel workbook (every sheet becomes a table) or CSV files (each becomes a table). Column names are made SQL-friendly, then **the same `connect()` and the same `ask()`** run on your data, step by step. Nothing leaves this Colab session. No file to hand? Use `sample_files/brightwave_sales.xlsx` from the workshop folder.

In [16]:
from google.colab import files

def load_spreadsheets(uploads, path="my_data.db"):
    '''Every CSV, and every sheet of an Excel workbook, becomes one table in a new SQLite file.'''
    target = sqlite3.connect(path)
    for name, blob in uploads.items():
        open(name, "wb").write(blob)
        sheets = {name.rsplit(".", 1)[0]: pd.read_csv(name)} if name.lower().endswith(".csv") else pd.read_excel(name, sheet_name=None)
        for sheet, frame in sheets.items():
            frame.columns = [re.sub(r"\W+", "_", str(c)).strip("_") for c in frame.columns]     # "Amount (GBP)" -> "Amount_GBP"
            frame.to_sql(re.sub(r"\W+", "_", sheet), target, if_exists="replace", index=False)
    return path

conn, db, catalogue = connect(load_spreadsheets(files.upload()))      # step 1 on YOUR tables; ask() now uses them


━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
 Step 1 · Catalogue
━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━
0 tables, 0 characters: []


In [17]:
while True:                                                                      # ask anything, 'quit' to stop
    q = input("Ask your data: ").strip()
    if q.lower() in ("quit", "exit", "q", ""): break
    ask(q)

Ask your data: q


If the SQL keeps failing on your file, read the banners in order: did step 2 pick the right table? Are the column names in step 3 what you expected after cleaning? What did the database say in step 6? Nine times out of ten the fix is a clearer column name or a sample row, not a better model.

## Exercises · Recap
1. **Self-check step**: between steps 4 and 5, ask the model *"Does this SQL answer the question? Reply yes or no and why"* and only run on yes.
2. **Bigger catalogue**: give `choose_tables` a one-line description per table (write them by hand) and compare its choices on the employee question.
3. **Chart**: `run_sql(sql).plot(x=..., y=..., kind="bar")` after step 5.
4. **Towards an agent**: after step 7, ask the model *"Do you need another query to answer fully?"* and loop if yes. That is Lab 3's pattern.

| Step | Who decides | Function | Library |
|---|---|---|---|
| 1 Catalogue | code | `connect()` → `SQLDatabase(...).table_info` | LangChain |
| 2 Choose tables | **model** | `choose_tables()` | `chat_json` |
| 3 Inject schema | code | `db.get_table_info(tables)` | LangChain |
| 4 Write SQL | **model** | `write_sql()` | `chat` |
| 5 Validate + run | code | `run_sql()` | sqlglot, pandas |
| 6 Retry with error | code | loop in `query()` | — |
| 7 Explain | **model** | `explain()` | `chat` |

**Next → Interactive:** an agent that decides which tool to call, with a human approving the risky ones.